In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 5, Finished, Available, Finished, False)

# Data Ingestion

This section extracts historical stock market data from Yahoo Finance and loads it into the Bronze layer.   

In [2]:
import yfinance as yf
import pandas as pd

from datetime import datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql.functions import *

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 6, Finished, Available, Finished, False)

In [3]:
stocks = {
    "Banking": ["HDFCBANK.NS", "ICICIBANK.NS"],
    "IT": ["TCS.NS", "INFY.NS"],
    "Energy": ["RELIANCE.NS", "ONGC.NS"],
    "Auto": ["TMPV.NS", "MARUTI.NS"],
    "Pharma": ["SUNPHARMA.NS", "CIPLA.NS"]
}

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 7, Finished, Available, Finished, False)

In [4]:
one_year_ago = (
    datetime.today() - timedelta(days=365)
).strftime("%Y-%m-%d")

print(one_year_ago)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 8, Finished, Available, Finished, False)

2025-06-05


In [5]:
all_data = []

for sector, tickers in stocks.items():

    for ticker in tickers:

        try:

            df = yf.download(
                ticker,
                start=one_year_ago,
                auto_adjust=False
            )

            if df.empty:

                print(f"No data for {ticker}")
                continue

            df.reset_index(inplace=True)

            df.columns = [
                col[0] if isinstance(col, tuple)
                else col
                for col in df.columns
            ]

            df["Ticker"] = ticker
            df["Sector"] = sector

            all_data.append(df)

            print(f"Downloaded {ticker}")

        except Exception as e:

            print(f"Error for {ticker}: {e}")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 9, Finished, Available, Finished, False)

[*********************100%***********************]  1 of 1 completed


Downloaded INFY.NS


In [6]:
final_df = pd.concat(
    all_data,
    ignore_index=True
)

print(final_df.shape)

final_df.head()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 10, Finished, Available, Finished, False)

(2500, 9)


,index,Adj Close,Close,High,Low,Open,Volume,Ticker,Sector
0,2025-06-05,961.799316,974.799988,979.450012,970.099976,974.000000,22255418,HDFCBANK.NS,Banking
1,2025-06-06,976.155273,989.349976,998.150024,971.349976,972.000000,30127008,HDFCBANK.NS,Banking
2,2025-06-09,976.254028,989.450012,997.500000,987.950012,997.500000,13202588,HDFCBANK.NS,Banking
3,2025-06-10,969.544739,982.650024,991.400024,980.549988,991.400024,15533126,HDFCBANK.NS,Banking
4,2025-06-11,962.194031,975.200012,983.799988,973.500000,983.799988,11096394,HDFCBANK.NS,Banking


In [7]:
spark_df = spark.createDataFrame(final_df)

spark_df.show(5)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 11, Finished, Available, Finished, False)

+-------------------+-----------------+-----------------+-----------------+-----------------+-----------------+--------+-----------+-------+
|              index|        Adj Close|            Close|             High|              Low|             Open|  Volume|     Ticker| Sector|
+-------------------+-----------------+-----------------+-----------------+-----------------+-----------------+--------+-----------+-------+
|2025-06-05 00:00:00|  961.79931640625|974.7999877929688|979.4500122070312|970.0999755859375|            974.0|22255418|HDFCBANK.NS|Banking|
|2025-06-06 00:00:00|   976.1552734375|989.3499755859375|998.1500244140625|971.3499755859375|            972.0|30127008|HDFCBANK.NS|Banking|
|2025-06-09 00:00:00|976.2540283203125|989.4500122070312|            997.5|987.9500122070312|            997.5|13202588|HDFCBANK.NS|Banking|
|2025-06-10 00:00:00|969.5447387695312|982.6500244140625|991.4000244140625|980.5499877929688|991.4000244140625|15533126|HDFCBANK.NS|Banking|
|2025-06-11 0

In [8]:
print(spark_df.columns)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 12, Finished, Available, Finished, False)

['index', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Sector']


In [9]:
spark_df = spark_df.withColumnRenamed(
    "index",
    "Trade_Date"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 13, Finished, Available, Finished, False)

In [10]:
spark_df = spark_df.withColumnRenamed(
    "Adj Close",
    "Adj_Close"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 14, Finished, Available, Finished, False)

In [11]:
for col_name in spark_df.columns:

    new_name = (
        col_name
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
    )

    spark_df = spark_df.withColumnRenamed(
        col_name,
        new_name
    )

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 15, Finished, Available, Finished, False)

In [12]:
spark_df = spark_df.dropDuplicates(
    ["Trade_Date", "Ticker"]
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 16, Finished, Available, Finished, False)

In [13]:
print(spark_df.columns)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 17, Finished, Available, Finished, False)

['Trade_Date', 'Adj_Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'Sector']


In [14]:
print(spark_df.count())

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 18, Finished, Available, Finished, False)

2500


# Bronze Layer

Raw market data is stored in the Bronze layer with duplicate prevention and rolling 1-year retention logic.

In [15]:
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("bronze.market_raw")

print("Bronze Loaded Successfully")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 19, Finished, Available, Finished, False)

Bronze Loaded Successfully


In [16]:
spark.sql("""
SELECT COUNT(*) AS row_count
FROM bronze.market_raw
""").show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 20, Finished, Available, Finished, False)

+---------+
|row_count|
+---------+
|     2500|
+---------+



In [17]:
spark.sql("""
SELECT *
FROM bronze.market_raw
LIMIT 5
""").show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 21, Finished, Available, Finished, False)

+-------------------+------------------+-----------------+-----------------+------------------+------------------+--------+------------+-------+
|         Trade_Date|         Adj_Close|            Close|             High|               Low|              Open|  Volume|      Ticker| Sector|
+-------------------+------------------+-----------------+-----------------+------------------+------------------+--------+------------+-------+
|2025-06-05 00:00:00|1460.3533935546875|1489.800048828125|1498.800048828125|            1475.0|            1476.0| 1570159|    CIPLA.NS| Pharma|
|2025-06-05 00:00:00|   961.79931640625|974.7999877929688|979.4500122070312| 970.0999755859375|             974.0|22255418| HDFCBANK.NS|Banking|
|2025-06-05 00:00:00|  1443.66064453125|1454.800048828125|           1459.0| 1427.300048828125|1430.9000244140625|11385668|ICICIBANK.NS|Banking|
|2025-06-05 00:00:00|   1530.8642578125|1554.300048828125|1567.699951171875|1541.5999755859375|1547.9000244140625| 6948539|     IN

In [18]:
bronze_df = spark.table(
    "bronze.market_raw"
)

silver_df = bronze_df.dropDuplicates(
    ["Trade_Date", "Ticker"]
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 22, Finished, Available, Finished, False)

In [19]:
spark.sql("""
SELECT
MIN(Trade_Date) as oldest_date,
MAX(Trade_Date) as latest_date,
COUNT(*) as total_rows
FROM bronze.market_raw
""").show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 23, Finished, Available, Finished, False)

+-------------------+-------------------+----------+
|        oldest_date|        latest_date|total_rows|
+-------------------+-------------------+----------+
|2025-06-05 00:00:00|2026-06-05 00:00:00|      2500|
+-------------------+-------------------+----------+



# Daily Stock Performance

Calculate daily returns and stock-level performance metrics.

In [20]:
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver.market_clean")

print("Silver Loaded")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 24, Finished, Available, Finished, False)

Silver Loaded


In [21]:
silver_df.printSchema()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 25, Finished, Available, Finished, False)

root
 |-- Trade_Date: timestamp (nullable = true)
 |-- Adj_Close: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- Sector: string (nullable = true)



In [22]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy("Ticker").orderBy("Trade_Date")

silver_returns_df = (
    silver_df
    .withColumn(
        "Prev_Close",
        F.lag("Close").over(window_spec)
    )
    .withColumn(
        "Daily_Return_Pct",
        (
            (F.col("Close") - F.col("Prev_Close"))
            / F.col("Prev_Close")
        ) * 100
    )
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 26, Finished, Available, Finished, False)

In [23]:
silver_returns_df.select(
    "Trade_Date",
    "Ticker",
    "Close",
    "Prev_Close",
    "Daily_Return_Pct"
).show(10)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 27, Finished, Available, Finished, False)

+-------------------+--------+------------------+------------------+-------------------+
|         Trade_Date|  Ticker|             Close|        Prev_Close|   Daily_Return_Pct|
+-------------------+--------+------------------+------------------+-------------------+
|2025-06-05 00:00:00|CIPLA.NS| 1489.800048828125|              NULL|               NULL|
|2025-06-06 00:00:00|CIPLA.NS| 1504.199951171875| 1489.800048828125| 0.9665661076516239|
|2025-06-09 00:00:00|CIPLA.NS|1506.5999755859375| 1504.199951171875|0.15955487913642838|
|2025-06-10 00:00:00|CIPLA.NS| 1510.800048828125|1506.5999755859375|0.27877826299273856|
|2025-06-11 00:00:00|CIPLA.NS| 1521.699951171875| 1510.800048828125| 0.7214655805845833|
|2025-06-12 00:00:00|CIPLA.NS| 1502.800048828125| 1521.699951171875| -1.242025560242347|
|2025-06-13 00:00:00|CIPLA.NS| 1505.199951171875| 1502.800048828125|0.15969538633043234|
|2025-06-16 00:00:00|CIPLA.NS|            1527.0| 1505.199951171875| 1.4483158075545077|
|2025-06-17 00:00:00|

In [24]:
from pyspark.sql.functions import min, max

spark.table("silver.market_clean").select(
    min("Trade_Date").alias("Oldest_Date"),
    max("Trade_Date").alias("Latest_Date")
).show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 28, Finished, Available, Finished, False)

+-------------------+-------------------+
|        Oldest_Date|        Latest_Date|
+-------------------+-------------------+
|2025-06-05 00:00:00|2026-06-05 00:00:00|
+-------------------+-------------------+



In [25]:
dim_stock = spark.sql("""
SELECT DISTINCT
    Ticker,
    Sector
FROM silver.market_clean
""")

dim_stock.write \
    .mode("overwrite") \
    .saveAsTable("gold.dim_stock")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 29, Finished, Available, Finished, False)

In [26]:
dim_date = spark.sql("""
SELECT DISTINCT
    Trade_Date
FROM silver.market_clean
""")

dim_date.write \
    .mode("overwrite") \
    .saveAsTable("gold.dim_date")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 30, Finished, Available, Finished, False)

# Daily Price Summary

In [27]:
from pyspark.sql.functions import *

gold_price_summary = (
    silver_df
    .groupBy("Trade_Date")
    .agg(
        avg("Close").alias("Avg_Close"),
        max("High").alias("Highest_Price"),
        min("Low").alias("Lowest_Price"),
        sum("Volume").alias("Total_Volume")
    )
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 31, Finished, Available, Finished, False)

In [28]:
gold_price_summary.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold.daily_price_summary")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 32, Finished, Available, Finished, False)

In [29]:
spark.sql("""
SELECT *
FROM gold.daily_price_summary
ORDER BY Trade_Date DESC
LIMIT 10
""").show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 33, Finished, Available, Finished, False)

+-------------------+------------------+-------------+------------------+------------+
|         Trade_Date|         Avg_Close|Highest_Price|      Lowest_Price|Total_Volume|
+-------------------+------------------+-------------+------------------+------------+
|2026-06-05 00:00:00|2359.2599853515626|      13175.0|263.70001220703125|    84649696|
|2026-06-04 00:00:00| 2366.194989013672|      13264.0| 265.8999938964844|   134874607|
|2026-06-03 00:00:00| 2364.674984741211|      13206.0|264.29998779296875|   133846068|
|2026-06-02 00:00:00| 2385.359997558594|      13051.0|261.04998779296875|   171261223|
|2026-06-01 00:00:00|2358.4299896240236|      13267.0|            263.25|   116337046|
|2026-05-29 00:00:00|2372.8449829101564|      13508.0|261.54998779296875|   386534897|
|2026-05-28 00:00:00|2412.7449951171875|      13364.0|274.04998779296875|           0|
|2026-05-27 00:00:00|2412.7449951171875|      13417.0|272.45001220703125|   152405805|
|2026-05-26 00:00:00|2399.7600006103517|   

# Daily Stock Performance

Calculate daily returns and stock-level performance metrics.

In [30]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, round

silver_df = spark.table("silver.market_clean")

w = Window.partitionBy("Ticker").orderBy("Trade_Date")

gold_daily_perf = (
    silver_df
    .withColumn("Previous_Close", lag("Close").over(w))
    .withColumn(
        "Daily_Return_Pct",
        round(
            ((col("Close") - col("Previous_Close"))
            / col("Previous_Close")) * 100,
            2
        )
    )
)

gold_daily_perf.write.mode("overwrite").saveAsTable(
    "gold.daily_stock_performance"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 34, Finished, Available, Finished, False)

# Sector Performance

Aggregate stock performance at sector level.

In [31]:
from pyspark.sql.functions import avg

sector_perf = (
    gold_daily_perf
    .groupBy("Trade_Date","Sector")
    .agg(
        round(
            avg("Daily_Return_Pct"),
            2
        ).alias("Sector_Return_Pct")
    )
)

sector_perf.write.mode("overwrite").saveAsTable(
    "gold.sector_daily_performance"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 35, Finished, Available, Finished, False)

# Volume Leaders

Identify stocks with unusually high trading volume.

In [32]:
volume_leaders = (
    silver_df
    .select(
        "Trade_Date",
        "Ticker",
        "Sector",
        "Volume"
    )
)

volume_leaders.write.mode("overwrite").saveAsTable(
    "gold.volume_leaders"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 36, Finished, Available, Finished, False)

# Moving Average 20 Days    

In [33]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

w = (
    Window
    .partitionBy("Ticker")
    .orderBy("Trade_Date")
    .rowsBetween(-19,0)
)

ma20 = (
    silver_df
    .withColumn(
        "MA20",
        avg("Close").over(w)
    )
)

ma20.write.mode("overwrite").saveAsTable(
    "gold.moving_average_20"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 37, Finished, Available, Finished, False)

# Momentum Signals

Generate BUY and SELL signals using moving average analysis.

In [34]:
from pyspark.sql.functions import when

signal_df = (
    ma20
    .withColumn(
        "Signal",
        when(
            col("Close") > col("MA20"),
            "BUY"
        ).otherwise("SELL")
    )
)

signal_df.write.mode("overwrite").saveAsTable(
    "gold.momentum_signal"
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 38, Finished, Available, Finished, False)

# Volatility Analysis

Calculate rolling 20-day volatility for each stock.

In [35]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

vol_window = (
    Window
    .partitionBy("Ticker")
    .orderBy("Trade_Date")
    .rowsBetween(-19, 0)
)

volatility_df = (
    silver_returns_df
    .withColumn(
        "Volatility_20D",
        F.stddev("Daily_Return_Pct").over(vol_window)
    )
    .select(
        "Trade_Date",
        "Ticker",
        "Sector",
        "Daily_Return_Pct",
        "Volatility_20D"
    )
)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 39, Finished, Available, Finished, False)

In [36]:
volatility_df.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold.volatility_signal")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 40, Finished, Available, Finished, False)

In [37]:
display(volatility_df)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8e13835-77f8-4dfb-8b54-c1d90f79fb70)

# Top Movers

In [38]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

rank_window = (
    Window
    .partitionBy("Trade_Date")
    .orderBy(F.desc("Daily_Return_Pct"))
)

top_movers_df = (
    silver_returns_df
    .withColumn(
        "Rank",
        F.rank().over(rank_window)
    )
    .select(
        "Trade_Date",
        "Ticker",
        "Sector",
        "Daily_Return_Pct",
        "Rank"
    )
)

top_movers_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.top_movers")

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 42, Finished, Available, Finished, False)

In [39]:
display(top_movers_df)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 228b8615-e37b-4318-b638-77fdcd31df1b)

In [40]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 44, Finished, Available, Finished, False)

+----------------------------------------------+------------------------+-----------+
|namespace                                     |tableName               |isTemporary|
+----------------------------------------------+------------------------+-----------+
|RetailTradingIntelligence.MarketLakehouse.gold|daily_price_summary     |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|daily_stock_performance |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|dim_date                |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|dim_stock               |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|momentum_signal         |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|moving_average_20       |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|sector_daily_performance|false      |
|RetailTradingIntelligence.MarketLakehouse.gold|top_movers              |false      |
|RetailTradingIntelligence.MarketLakehouse.gold|volati

In [41]:
spark.sql("""
SELECT
COUNT(DISTINCT Trade_Date) as Trading_Days
FROM silver.market_clean
""").show()

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 45, Finished, Available, Finished, False)

+------------+
|Trading_Days|
+------------+
|         250|
+------------+



In [42]:
spark.sql("""
SELECT
Trade_Date,
COUNT(*) as Rows_Per_Day
FROM silver.market_clean
GROUP BY Trade_Date
ORDER BY Trade_Date DESC
""").show(20, False)

StatementMeta(, 17b83af1-fb96-4aa6-80d2-e545c5bc4d48, 46, Finished, Available, Finished, False)

+-------------------+------------+
|Trade_Date         |Rows_Per_Day|
+-------------------+------------+
|2026-06-05 00:00:00|10          |
|2026-06-04 00:00:00|10          |
|2026-06-03 00:00:00|10          |
|2026-06-02 00:00:00|10          |
|2026-06-01 00:00:00|10          |
|2026-05-29 00:00:00|10          |
|2026-05-28 00:00:00|10          |
|2026-05-27 00:00:00|10          |
|2026-05-26 00:00:00|10          |
|2026-05-25 00:00:00|10          |
|2026-05-22 00:00:00|10          |
|2026-05-21 00:00:00|10          |
|2026-05-20 00:00:00|10          |
|2026-05-19 00:00:00|10          |
|2026-05-18 00:00:00|10          |
|2026-05-15 00:00:00|10          |
|2026-05-14 00:00:00|10          |
|2026-05-13 00:00:00|10          |
|2026-05-12 00:00:00|10          |
|2026-05-11 00:00:00|10          |
+-------------------+------------+
only showing top 20 rows

